
# EdgeAI Assignment — SVDF Wake-Word Detection for OpenMV

## Objective

In this assignment, you will:

1. Build a wake-word detector using an **SVDF (Singular Value Decomposition Filter)** neural network.
2. Train the model on multiple wake words:
   - `"Hey Jarvis"`
   - `"OK Nambu"`
   - additional custom wake words (optional)
3. Create a strong `"unknown"` / `"no wake word"` class.
4. Export the trained model to **TensorFlow Lite**.
5. Prepare the model for deployment on the **OpenMV platform**.
6. Evaluate accuracy, latency, model size, and robustness.
7. Complete a self-grading rubric at the end.

---

# Important Design Goal

A wake-word detector fails more often from **false triggers** than from missed detections.

Therefore:

- The `"unknown"` class is CRITICAL.
- Your model should learn:
  - what the wake word sounds like
  - AND what normal speech/background noise sounds like

A weak unknown class will cause:
- random triggering
- TV-triggering
- conversation-triggering
- poor real-world performance

Your unknown dataset should be MUCH larger than the wake-word dataset.

Recommended ratio:

| Class | Suggested Amount |
|---|---|
| Wake Word | 1x |
| Unknown / Background | 5x–20x |
| Silence / Noise | 2x–5x |

---

# Hardware Target

Target platform:
- OpenMV H7 Plus / RT / N6
- TensorFlow Lite Micro deployment
- Microphone audio streaming
- Real-time inference

---

# Recommended Development Environment

- Python 3.10+
- TensorFlow 2.15+
- Jupyter Notebook
- librosa
- numpy
- pandas
- matplotlib
- kagglehub (optional)

---

# Deliverables

You must submit:

1. Completed notebook
2. Trained `.tflite` model
3. Accuracy metrics
4. Confusion matrix
5. Screenshot/video of OpenMV deployment
6. Written discussion of:
   - false positives
   - false negatives
   - optimizations attempted




# Part 1 — Install Dependencies

## Goal

Install all required packages for:
- audio processing
- TensorFlow training
- feature extraction
- TensorFlow Lite conversion
- visualization

## Student Tasks

- Run the cell below
- Resolve any missing dependencies
- Restart the notebook kernel if necessary


In [1]:

# Uncomment if packages are missing

!pip install tensorflow librosa soundfile pandas matplotlib scikit-learn kagglehub



# Part 2 — Kaggle Dataset References

## Goal

You must collect:
1. wake-word examples
2. unknown speech
3. silence/background noise

---

# Suggested Kaggle Datasets

## Speech Commands Dataset

Contains:
- yes
- no
- up
- down
- stop
- go
- many speakers

Excellent for:
- unknown class
- transfer learning
- augmentation

Dataset:
https://www.kaggle.com/datasets/jbuchner/speech-commands

---

## Wake Word Dataset Examples

### Hey Jarvis Dataset
https://www.kaggle.com/search?q=hey+jarvis+wake+word

### Custom Wake Word Datasets
https://www.kaggle.com/search?q=wake+word

---

## Environmental Noise

Use environmental sounds for robustness.

Examples:
- office noise
- keyboard clicks
- traffic
- fans
- HVAC
- TV audio

Dataset:
https://www.kaggle.com/datasets/mmoreaux/environmental-sound-classification-50

---

# IMPORTANT

Your `"unknown"` class should contain:

- random speech
- partial wake words
- other languages
- coughs
- music
- silence
- background noise

A model trained ONLY on clean wake words will fail badly in the real world.



# Part 3 — Define Dataset Structure

## Goal

Create a clean dataset layout.

Recommended structure:

```text
dataset/
    hey_jarvis/
    ok_nambu/
    unknown/
    silence/
```

Each folder should contain:
- `.wav` audio files
- mono audio
- consistent sample rate

---

# Audio Format Requirements

Use:
- 16 kHz sample rate
- mono
- 16-bit PCM
- WAV format

This format matches:
- TensorFlow Lite Micro examples
- OpenMV microphone capture pipeline


In [ ]:

from pathlib import Path

DATASET_DIR = Path("dataset")

WAKE_WORDS = [
    "hey_jarvis",
    "ok_nambu"
]

UNKNOWN_CLASS = "unknown"
SILENCE_CLASS = "silence"

print("Dataset root:", DATASET_DIR)
print("Wake words:", WAKE_WORDS)



# Part 4 — Audio Configuration

## Goal

Define all audio parameters.

These MUST match:
- training
- preprocessing
- inference
- OpenMV deployment

Changing parameters later can invalidate the trained model.


In [ ]:

SAMPLE_RATE = 16000
CLIP_DURATION_MS = 1000
WINDOW_SIZE_MS = 30
WINDOW_STRIDE_MS = 20
FEATURE_BIN_COUNT = 40

print("Sample rate:", SAMPLE_RATE)
print("Clip duration:", CLIP_DURATION_MS)
print("Feature bins:", FEATURE_BIN_COUNT)



# Part 5 — Load Audio Files

## Goal

Load and normalize all audio clips.

## Student Tasks

Implement:
- recursive dataset loading
- WAV parsing
- label assignment
- normalization

## Requirements

Each loaded clip should:
- have consistent length
- be mono
- be float32
- be normalized to [-1, 1]

## Suggested Libraries

- librosa
- soundfile
- scipy


In [ ]:

import librosa
import numpy as np

def load_audio_file(path, sample_rate=SAMPLE_RATE):
    '''
    STUDENT TODO:
    - Load WAV file
    - Resample if needed
    - Normalize audio
    - Pad or truncate to 1 second
    '''

    # Example placeholder:
    audio = np.zeros(sample_rate, dtype=np.float32)

    return audio



# Part 6 — Feature Extraction

## Goal

Convert raw audio into ML-friendly features.

Recommended:
- MFCCs
- log-mel spectrograms

*MFCC stands for Mel-frequency Cepstral Coefficients. It’s a feature used in automatic speech and speaker recognition. Essentially, it’s a way to represent the short-term power spectrum of a sound which helps machines understand and process human speech more effectively. Imagine your voice as a unique fingerprint. MFCCs, function similarly to a unique code capturing the salient features of your speech and enabling computers to discern between distinct words, and sounds. In speech recognition applications where computers must translate spoken words into text this code is especially helpful.*

**SVDF models perform well on compact temporal features.**

---

# Why Feature Extraction Matters

The model should learn:
- phoneme timing
- spectral shape
- speech rhythm

rather than raw waveform noise.

---

# Recommended Output Shape

Example:
- time frames × feature bins
- `(49, 40)`


In [ ]:

def extract_features(audio):

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=SAMPLE_RATE,
        n_mfcc=N_MFCC,
        n_fft=int(SAMPLE_RATE * WINDOW_SIZE_MS / 1000),
        hop_length=int(SAMPLE_RATE * WINDOW_STRIDE_MS / 1000)
    )

    mfcc = mfcc.T

    # normalize features
    mfcc = (mfcc - np.mean(mfcc)) / (np.std(mfcc) + 1e-6)

    return mfcc.astype(np.float32)



# Part 7 — Dataset Construction

## Goal

Create:
- feature tensors
- label vectors
- train/validation/test splits

---

# IMPORTANT

Your dataset must be balanced carefully.

A GOOD deployment dataset should include:

| Scenario | Included? |
|---|---|
| Quiet room | YES |
| Background TV | YES |
| Multiple speakers | YES |
| Partial wake words | YES |
| Noise bursts | YES |
| Silence | YES |

---

# Student Tasks

Implement:
- feature extraction loop
- label encoding
- train/test split
- shuffling


In [ ]:

X = []
y = []

'''
STUDENT TODO:
- Walk through dataset folders
- Load audio
- Extract features
- Append to X (audio) and y (features)
'''

print("Dataset shape:", X.shape)
print("Labels:", y.shape)


# Encode Labels and Split Dataset


In [ ]:

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)



# Visualize Example MFCC


In [ ]:

plt.figure(figsize=(10, 4))

plt.imshow(X_train[0].T, aspect='auto', origin='lower')

plt.title("Example MFCC")
plt.xlabel("Time")
plt.ylabel("MFCC Bin")

plt.colorbar()
plt.show()



# Part 8 — Build the SVDF Model

## Goal

Create a compact SVDF architecture.

SVDF is useful because:
- low RAM usage
- efficient temporal modeling
- ideal for microcontrollers
- used in embedded speech systems

---

# Suggested Architecture

Input:
- `(time_steps, feature_bins)`

Layers:
1. SVDF-compatible dense projection
2. Temporal filtering
3. Dense classifier
4. Softmax output

---

# Constraints

Target:
- small memory footprint
- low inference latency
- TFLite-compatible operations


In [ ]:

import tensorflow as tf
from tensorflow.keras import layers

NUM_CLASSES = 4  # hey_jarvis, ok_nambu, unknown, silence

'''
STUDENT TODO:
Build SVDF-style model.

Possible ideas:
- Dense projection
- Depthwise temporal operations
- Small recurrent approximation
- Low-rank layers
'''
time_steps = X_train.shape[1]
feature_bins = X_train.shape[2]

model = tf.keras.Sequential([
    layers.Input(shape=(49, FEATURE_BIN_COUNT)),

    # Frequency projection. WHY?
    layers.Dense(64, activation='relu'),

    # Temporal filtering
    layers.DepthwiseConv1D(
        kernel_size=5,
        padding='same',
        activation='relu'
    ),

    layers.SeparableConv1D(
        filters=32,
        kernel_size=3,
        padding='same',
        activation='relu'
    ),

    layers.GlobalAveragePooling1D(),

    layers.Dense(32, activation='relu'),

    layers.Dropout(0.2),

    layers.Flatten(),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.summary()



# Part 9 — Compile and Train

## Goal

Train the wake-word detector.

---

# Recommended Training Features

- early stopping
- learning rate scheduling
- checkpoint saving

---

# Important

Monitor:
- validation accuracy
- false positives
- unknown class confusion

Accuracy alone is NOT enough.


In [ ]:

'''
STUDENT TODO:
- Compile model
- Select optimizer
- Select loss function
- Train model
'''

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        patience=5,
        restore_best_weights=True
    )
]

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=32,
    callbacks=callbacks
)




# Plot Training Curves


In [ ]:

plt.figure(figsize=(8,4))
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.title("Training Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend(["Train", "Validation"])

plt.show()



# Part 10 — Evaluate the Model

## Goal

Measure:
- accuracy
- confusion matrix
- robustness

---

# Important Questions

1. Does the model falsely trigger?
2. Does background speech activate it?
3. Does it confuse wake words?
4. How does it behave in noise?


In [ ]:

from sklearn.metrics import confusion_matrix, classification_report

'''
STUDENT TODO:
- Run predictions
- Generate confusion matrix
- Print classification report
'''
test_loss, test_acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", test_acc)

predictions = model.predict(X_test)
pred_labels = np.argmax(predictions, axis=1)
print(classification_report(
    y_test,
    pred_labels,
    target_names=encoder.classes_
))

cm = confusion_matrix(y_test, pred_labels)
plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    xticklabels=encoder.classes_,
    yticklabels=encoder.classes_
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()



# False Positive Analysis

This is extremely important for wake-word systems.

We inspect confidence values to determine whether:
- unknown speech activates wake words
- thresholds should be adjusted


In [ ]:

wake_indices = [
    encoder.transform(["hey_jarvis"])[0],
    encoder.transform(["ok_nambu"])[0]
]

for i in range(min(10, len(predictions))):

    probs = predictions[i]

    max_wake_conf = max(probs[wake_indices])

    print(
        f"Sample {i} | "
        f"True={encoder.inverse_transform([y_test[i]])[0]} | "
        f"Wake Confidence={max_wake_conf:.3f}"
    )



# Part 11 — TensorFlow Lite Export

## Goal

Convert the trained model into:
- `.tflite`
- TensorFlow Lite Micro compatible format

This is REQUIRED for OpenMV deployment.


In [ ]:

'''
STUDENT TODO:
Convert model to TensorFlow Lite
'''

converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Optional optimizations
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open("wake_word_svdf.tflite", "wb") as f:
    f.write(tflite_model)

print("Saved wake_word_svdf.tflite")



# Part 12 — Optional Quantization

## Goal

Reduce:
- model size
- RAM usage
- inference latency

Quantization is VERY important for microcontrollers.

---

# Recommended Modes

| Mode | Benefits |
|---|---|
| Float16 | Smaller model |
| INT8 | Fastest MCU inference |
| Dynamic Range | Easy optimization |

---

# Challenge

Compare:
- accuracy before quantization
- accuracy after quantization


In [ ]:

'''
OPTIONAL STUDENT TODO:
Implement:
- INT8 quantization
- representative dataset
- benchmarking
'''
def representative_dataset():
    for i in range(min(100, len(X_train))):
        yield [X_train[i:i+1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

converter.representative_dataset = representative_dataset

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]

converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

quant_model = converter.convert()

with open("wake_word_svdf_int8.tflite", "wb") as f:
    f.write(quant_model)

print("Saved quantized model")
print("Quantized size KB:", len(quant_model) / 1024)


# TensorFlow Lite Inference Test


In [ ]:

interpreter = tf.lite.Interpreter(
    model_path="wake_word_svdf.tflite"
)

interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

sample = X_test[0:1].astype(np.float32)

interpreter.set_tensor(
    input_details[0]['index'],
    sample
)

interpreter.invoke()

output = interpreter.get_tensor(
    output_details[0]['index']
)

print("Output:", output)
print("Predicted:", encoder.inverse_transform([np.argmax(output)]))



# Part 13 — OpenMV Deployment Mapping

## Goal

Prepare the model for deployment on OpenMV.

---

# OpenMV Audio Capture Format

Recommended microphone configuration:

| Parameter | Value |
|---|---|
| Sample Rate | 16 kHz |
| Channels | Mono |
| Sample Type | int16 |
| Clip Duration | 1 second |

---

# Why This Matters

Your training pipeline MUST match:
- inference sample rate
- frame size
- feature extraction
- normalization

Mismatch = failed inference.

---

# Deployment Pipeline

OpenMV Device:
1. capture microphone audio
2. compute spectrogram/MFCC
3. run TFLite model
4. apply confidence threshold
5. trigger action

---

# Confidence Threshold

Typical values:
- 0.70
- 0.80
- 0.90

Higher threshold:
- fewer false triggers
- more missed detections


In [ ]:

# Example pseudo-code for OpenMV deployment

'''
import audio
import tf

model = tf.load("wake_word_svdf.tflite")

while True:
    samples = capture_audio()

    features = compute_features(samples)

    prediction = model.predict(features)

    if prediction["hey_jarvis"] > 0.85:
        print("Wake word detected")
'''



# Part 14 — Benchmarking and Optimization

## Goal

Measure:
- model size
- inference speed
- memory usage
- false trigger rate

---

# Suggested Experiments

1. Change MFCC count
2. Change window stride
3. Add more unknown data
4. Add synthetic noise
5. Compare quantized vs non-quantized

---

# Engineering Tradeoffs

| Smaller Model | Larger Model |
|---|---|
| Faster | More accurate |
| Lower RAM | Higher RAM |
| Lower power | Better robustness |



# Part 15 — Final Reflection Questions

Answer the following:

1. What caused the most false positives?
2. How important was the unknown class?
3. What deployment constraints mattered most?
4. What improvements would you try next?
5. Did quantization hurt accuracy?
6. Which wake word was easiest to detect?
7. Which environmental condition caused failures?



# Part 16 — Self-Grading Rubric

## Total: 100 Points

| Category | Points |
|---|---|
| Dataset quality | 15 |
| Unknown/noise class quality | 20 |
| Feature extraction implementation | 10 |
| SVDF architecture | 15 |
| Training process | 10 |
| Evaluation and analysis | 10 |
| TensorFlow Lite export | 5 |
| OpenMV deployment readiness | 10 |
| Optimization experiments | 5 |

---

# Instructions

Update the values in the next cell honestly.

Your score will be computed automatically.


In [ ]:

# SELF-GRADING CELL

dataset_quality = 0
unknown_class_quality = 0
feature_extraction = 0
svdf_architecture = 0
training_process = 0
evaluation_analysis = 0
tflite_export = 0
openmv_readiness = 0
optimization_experiments = 0

score = (
    dataset_quality +
    unknown_class_quality +
    feature_extraction +
    svdf_architecture +
    training_process +
    evaluation_analysis +
    tflite_export +
    openmv_readiness +
    optimization_experiments
)

print("=" * 50)
print("SELF-GRADING REPORT")
print("=" * 50)
print(f"Dataset Quality: {dataset_quality}/15")
print(f"Unknown Class Quality: {unknown_class_quality}/20")
print(f"Feature Extraction: {feature_extraction}/10")
print(f"SVDF Architecture: {svdf_architecture}/15")
print(f"Training Process: {training_process}/10")
print(f"Evaluation & Analysis: {evaluation_analysis}/10")
print(f"TFLite Export: {tflite_export}/5")
print(f"OpenMV Readiness: {openmv_readiness}/10")
print(f"Optimization Experiments: {optimization_experiments}/5")
print("-" * 50)
print(f"FINAL SCORE: {score}/100")

if score >= 90:
    print("Excellent work.")
elif score >= 75:
    print("Good work.")
elif score >= 60:
    print("Passing, but improvements needed.")
else:
    print("Insufficient completion.")
